# 20 · Real Course Retail Market Basket — Apriori + API Verification

**Run this notebook in VS Code** with the same `.venv` kernel you've used for notebooks 15–19.

- Uses the professor's **original 1,000-row CSV** (`transaction_id,item_1,…,item_5`).
- Reuses **14_market_basket_engine.ipynb**, with *no edits* to notebooks 00–19.
- Creates a **separate, fixed project ID for real course data**, so notebook 15 and notebook 16 can continue using synthetic fixtures without erasing the real data.
- Derives product names and SKUs from source strings, checks duplicates, reports Apriori support / confidence / lift, and tests the REST API.
- Saves an import audit containing the SHA-256 of the original CSV. Re-running is **idempotent**: identical imported transactions are not reinserted.

> **Data integrity:** The CSV is professor-supplied transaction-like data. Its authenticity as real customer purchases isn't independently verified. Association is not proof of technical compatibility. The exact-cart recommender abstains if its observed association rules don't meet configured thresholds.

**CSV location:** `data/course_sources/official_pdfs/Retail 2/CMPE256_Hackathon_market_basket_analysis_Release.csv`.

In [1]:
%run 14_market_basket_engine.ipynb

Database initialized: /Users/nehlinshanila/Downloads/group2-course-rag-engine/.runtime/group2.sqlite3
Apriori basket recommender ready; call seed_market_basket_demo() to load synthetic transactions.


In [2]:
import hashlib
import re
from collections import Counter
from pathlib import Path

REAL_RETAIL_PROJECT_ID = '22222222-2222-4222-8222-222222222222'
RETAIL_FILENAME = 'CMPE256_Hackathon_market_basket_analysis_Release.csv'
REPLACE_DIFFERENT_REAL_DATA = False  # Set True ONLY when intentionally replacing a changed real-data import.

RETAIL_CANDIDATES = [
    ROOT / 'data' / 'course_sources' / 'official_pdfs' / 'Retail 2' / RETAIL_FILENAME,
    ROOT / 'data' / 'course_sources' / 'official_pdfs' / 'Retail' / RETAIL_FILENAME,
    ROOT / 'data' / 'course_sources' / RETAIL_FILENAME,
]
retail_csv = next((p for p in RETAIL_CANDIDATES if p.is_file()), None)
if retail_csv is None:
    raise FileNotFoundError(
        'The course Retail CSV was not found. Place it at: ' + str(RETAIL_CANDIDATES[0])
    )

ITEM_PATTERN = re.compile(r'^(.+?)\s*\(SKU:\s*([^()]+?)\s*\)\s*$', re.IGNORECASE)

def parse_course_csv(path: Path):
    """Read the professor's wide transaction CSV, validate source values, return BasketRow-compatible lines."""
    parsed = []
    baskets_by_id = {}
    sku_catalog = {}
    basket_sizes = Counter()
    seen_ids = set()
    with path.open('r', encoding='utf-8-sig', newline='') as fh:
        reader = csv.DictReader(fh)
        header = reader.fieldnames or []
        expected = ['transaction_id'] + [f'item_{i}' for i in range(1, 6)]
        if header != expected:
            raise ValueError(f'Unexpected retail CSV schema: {header}; expected {expected}')
        for line, row in enumerate(reader, start=2):
            if None in row:
                raise ValueError(f'Extra CSV columns on line {line}')
            transaction_id = (row['transaction_id'] or '').strip()
            if not transaction_id:
                raise ValueError(f'Missing transaction_id on line {line}')
            if transaction_id in seen_ids:
                raise ValueError(f'Duplicate transaction_id: {transaction_id} (line {line})')
            seen_ids.add(transaction_id)
            one_basket = set()
            for col in expected[1:]:
                cell = (row[col] or '').strip()
                if not cell:
                    continue
                match = ITEM_PATTERN.fullmatch(cell)
                if not match:
                    raise ValueError(f'Unparseable item at line {line}, {col}: {cell!r}')
                product_name, sku = (match.group(1).strip(), match.group(2).strip())
                if sku in one_basket:
                    continue  # presence/absence market-basket semantics: same SKU once per transaction
                if sku in sku_catalog and sku_catalog[sku] != product_name:
                    raise ValueError(f'SKU {sku!r} has conflicting source product names')
                sku_catalog[sku] = product_name
                one_basket.add(sku)
                parsed.append({
                    'transaction_id': transaction_id,
                    'sku': sku,
                    'product_name': product_name,
                    'category': 'Unclassified retail product',  # professor CSV has no category column
                    'quantity': 1,
                })
            if not one_basket:
                raise ValueError(f'Empty transaction at CSV line {line}')
            baskets_by_id[transaction_id] = one_basket
            basket_sizes[len(one_basket)] += 1
    if not parsed:
        raise ValueError('No retail transaction items in source CSV')
    return parsed, baskets_by_id, sku_catalog, basket_sizes

source_rows, csv_baskets, source_catalog, sizes = parse_course_csv(retail_csv)
source_sha256 = hashlib.sha256(retail_csv.read_bytes()).hexdigest()
assert len(csv_baskets) == 1000 and len(source_catalog) == 40, 'Unexpected course CSV counts; inspect source'
print(json.dumps({
    'file': str(retail_csv.relative_to(ROOT)),
    'sha256': source_sha256,
    'transactions': len(csv_baskets),
    'distinct_products': len(source_catalog),
    'basket_lines': len(source_rows),
    'basket_sizes': dict(sorted(sizes.items())),
    'source_provenance': 'course_provided_csv_unverified_transaction_origin'
}, indent=2))

{
  "file": "data/course_sources/official_pdfs/Retail 2/CMPE256_Hackathon_market_basket_analysis_Release.csv",
  "sha256": "3d48b2480bee2ca3942f7add047233eb69461d210b996733c3eff3ee0bf5cbe6",
  "transactions": 1000,
  "distinct_products": 40,
  "basket_lines": 3575,
  "basket_sizes": {
    "2": 237,
    "3": 228,
    "4": 258,
    "5": 277
  },
  "source_provenance": "course_provided_csv_unverified_transaction_origin"
}


In [3]:
def indexed_baskets_by_id(project_id):
    pid = ensure_project_id(project_id)
    basket_dict = {}
    with db_connect() as cx:
        rows = cx.execute(
            'SELECT transaction_id, sku FROM basket_lines WHERE project_id=? ORDER BY transaction_id, sku',
            (pid,)
        ).fetchall()
    for row in rows:
        basket_dict.setdefault(row['transaction_id'], set()).add(row['sku'])
    return basket_dict

existing = indexed_baskets_by_id(REAL_RETAIL_PROJECT_ID)
if existing == csv_baskets:
    load_action = 'ALREADY_INDEXED_SAME_TRANSACTIONS'
    # The source file may have different descriptive metadata; check too.
    loaded_products = {p['sku']: p['name'] for p in basket_inventory(REAL_RETAIL_PROJECT_ID)['products']}
    if loaded_products != source_catalog:
        raise RuntimeError('Same SKU baskets but different product names: inspect source or allow explicit replacement.')
elif existing and not REPLACE_DIFFERENT_REAL_DATA:
    raise RuntimeError(
        'The real-data project already has DIFFERENT transactions. No changes were made. '
        'Review the CSV and set REPLACE_DIFFERENT_REAL_DATA=True only when intentional.'
    )
else:
    outcome = import_basket_rows(
        REAL_RETAIL_PROJECT_ID, source_rows,
        source_type='USER_DATASET', replace=bool(existing)
    )
    load_action = 'REPLACED_DIFFERENT_TRANSACTIONS' if existing else 'IMPORTED'
    print('Import result:', json.dumps(outcome, indent=2))

inventory = basket_inventory(REAL_RETAIL_PROJECT_ID)
assert inventory['transaction_count'] == len(csv_baskets)
assert len(inventory['products']) == len(source_catalog)
assert inventory['source_types'] == ['USER_DATASET']
assert not inventory['is_synthetic']
assert indexed_baskets_by_id(REAL_RETAIL_PROJECT_ID) == csv_baskets

audit = {
    'action': load_action,
    'project_id': REAL_RETAIL_PROJECT_ID,
    'csv_path': str(retail_csv.relative_to(ROOT)),
    'csv_sha256': source_sha256,
    'transaction_count': len(csv_baskets),
    'distinct_skus': len(source_catalog),
    'basket_lines': len(source_rows),
    'source_type': 'USER_DATASET',
    'provenance': 'course_provided_csv_unverified_transaction_origin',
}
audit_file = ROOT / '.runtime' / 'real_retail_import_audit.json'
audit_file.parent.mkdir(parents=True, exist_ok=True)
audit_file.write_text(json.dumps(audit, indent=2), encoding='utf-8')
print('PASS: course CSV imported or verified without modifying the synthetic fixture project')
print(json.dumps(audit, indent=2))

Import result: {
  "project_id": "22222222-2222-4222-8222-222222222222",
  "transaction_count": 1000,
  "product_count": 40,
  "line_count": 3575,
  "source_type": "USER_DATASET",
  "is_synthetic": false
}
PASS: course CSV imported or verified without modifying the synthetic fixture project
{
  "action": "IMPORTED",
  "project_id": "22222222-2222-4222-8222-222222222222",
  "csv_path": "data/course_sources/official_pdfs/Retail 2/CMPE256_Hackathon_market_basket_analysis_Release.csv",
  "csv_sha256": "3d48b2480bee2ca3942f7add047233eb69461d210b996733c3eff3ee0bf5cbe6",
  "transaction_count": 1000,
  "distinct_skus": 40,
  "basket_lines": 3575,
  "source_type": "USER_DATASET",
  "provenance": "course_provided_csv_unverified_transaction_origin"
}


## Association rules on this actual CSV

The synthetic demo used stronger associations. In the professor's CSV, even the strongest product pairs appear relatively infrequently. The **0.40 minimum confidence** used in notebook 15 is too strict for this dataset: the test below instead uses **minimum support 0.01 (at least 10 of 1,000 orders), confidence 0.10, and lift 1.05**. These are demonstration settings, not tuned production thresholds.

With these conservative thresholds, a cart containing two products may have no supported 2→1 rule. The engine will correctly abstain rather than fabricate a recommendation.

In [4]:
REAL_SETTINGS = dict(top_k=5, min_support=0.01, min_confidence=0.10, min_lift=1.05, max_itemset_size=4)
real_baskets = get_baskets(REAL_RETAIL_PROJECT_ID)
frequent = apriori_frequent_itemsets(real_baskets,
                                   min_support=REAL_SETTINGS['min_support'],
                                   max_itemset_size=REAL_SETTINGS['max_itemset_size'])
rules = association_rules(frequent, len(real_baskets),
                          min_confidence=REAL_SETTINGS['min_confidence'],
                          min_lift=REAL_SETTINGS['min_lift'])
print('Transaction count:', len(real_baskets))
print('Frequent itemsets:', len(frequent))
print('Association rules passing thresholds:', len(rules))
print('Antecedent sizes:', dict(sorted(Counter(len(r['antecedent']) for r in rules).items())))
if not rules:
    raise AssertionError('Unexpectedly no rules at the selected thresholds; inspect source data.')
print('\nTop 8 learned association rules (source transaction data, not compatibility evidence):')
for rule in rules[:8]:
    print(f"  {rule['antecedent']} -> {rule['consequent']}: confidence={rule['confidence']:.1%}, "
          f"lift={rule['lift']:.2f}, support={rule['support']:.1%}, "
          f"joint orders={rule['joint_transactions']}")

Transaction count: 1000
Frequent itemsets: 147
Association rules passing thresholds: 199
Antecedent sizes: {1: 199}

Top 8 learned association rules (source transaction data, not compatibility evidence):
  ['5800CO'] -> WS4916: confidence=20.0%, lift=2.25, support=1.5%, joint orders=15
  ['NDN-832V03-IP'] -> NVR-5500-16A00: confidence=17.3%, lift=1.92, support=1.4%, joint orders=14
  ['QNV-6010R'] -> NVR4208-8P-4KS2: confidence=16.9%, lift=1.61, support=1.4%, joint orders=14
  ['WS4916'] -> 5800CO: confidence=16.9%, lift=2.25, support=1.5%, joint orders=15
  ['DH-IPC-HDBW4431R-ZS'] -> WS4939: confidence=16.3%, lift=1.62, support=1.6%, joint orders=16
  ['WS4939'] -> DH-IPC-HDBW4431R-ZS: confidence=15.8%, lift=1.62, support=1.6%, joint orders=16
  ['NVR-5500-16A00'] -> NDN-832V03-IP: confidence=15.6%, lift=1.92, support=1.4%, joint orders=14
  ['DH-IPC-HFW4431R-Z'] -> 0407-001: confidence=15.3%, lift=1.70, support=1.3%, joint orders=13


In [5]:
from IPython.display import display, Markdown

candidate = next(rule for rule in rules if len(rule['antecedent']) == 1)
first_sku = candidate['antecedent'][0]
second_sku = candidate['consequent']

def show_real_cart(skus):
    result = market_basket_recommend(REAL_RETAIL_PROJECT_ID, skus, **REAL_SETTINGS)
    display(Markdown('#### Cart: ' + (', '.join(skus) if skus else '(empty)')))
    if result['abstained']:
        print('No supported recommendation:', result['reason'])
    else:
        for pos, item in enumerate(result['recommendations'], start=1):
            print(f"{pos}. {item['product_name']} [{item['sku']}] | "
                  f"confidence {item['confidence']:.1%} | "
                  f"lift {item['lift']:.2f} | support {item['support']:.1%} | "
                  f"joint transactions {item['evidence']['joint_transactions']}")
    return result

empty_result = show_real_cart([])
single_result = show_real_cart([first_sku])
pair_result = show_real_cart([first_sku, second_sku])
unknown_result = show_real_cart(['UNKNOWN-SKU'])
assert empty_result['reason'] == 'empty_cart'
assert not single_result['abstained'] and single_result['recommendations']
assert unknown_result['reason'] == 'unknown_sku'
assert pair_result['cart'] == [first_sku, second_sku]
print('\nPASS: each cart update is recomputed from the professor CSV; unsupported combinations abstain.')

#### Cart: (empty)

No supported recommendation: empty_cart


#### Cart: 5800CO

1. DSC WS4916 Smoke Detector [WS4916] | confidence 20.0% | lift 2.25 | support 1.5% | joint transactions 15


#### Cart: 5800CO, WS4916

No supported recommendation: no_supported_association_rules


#### Cart: UNKNOWN-SKU

No supported recommendation: unknown_sku

PASS: each cart update is recomputed from the professor CSV; unsupported combinations abstain.


In [6]:
# Optional interactive real-data cart inside VS Code. No external browser required.
try:
    import ipywidgets as widgets
    options = [(f"{p['name']} [{p['sku']}]", p['sku']) for p in inventory['products']]
    picker = widgets.SelectMultiple(options=options, rows=12,
                                    description='Cart:', layout=widgets.Layout(width='780px'))
    feedback = widgets.Output()

    def on_selection(change):
        with feedback:
            feedback.clear_output(wait=True)
            outcome = market_basket_recommend(REAL_RETAIL_PROJECT_ID, list(change['new']), **REAL_SETTINGS)
            print('Source: professor CSV | transactions:', outcome['training_transactions'])
            if outcome['abstained']:
                print('No supported recommendation:', outcome['reason'])
            else:
                for item in outcome['recommendations']:
                    print(f"{item['product_name']} | confidence={item['confidence']:.1%}, "
                          f"lift={item['lift']:.2f}, support={item['support']:.1%}")

    picker.observe(on_selection, names='value')
    display(widgets.VBox([widgets.HTML('<b>Select items to get real CSV-based recommendations</b>'),
                          picker, feedback]))
except ImportError:
    print('Interactive cart optional: install ipywidgets if wanted. All calculations above work without it.')

## Live FastAPI contract test (localhost developer mode)

The existing routes do not change. Use the *real* project UUID below in Group 3/Group 1, rather than the synthetic project UUID used by notebook 15:

- `GET /api/v1/projects/22222222-2222-4222-8222-222222222222/market-basket/products`
- `POST /api/v1/projects/22222222-2222-4222-8222-222222222222/market-basket/recommend`

The API is **development-only** until Group 3 integrates authentication and authorization.

In [7]:
import os
os.environ['GROUP2_DEV_MODE'] = 'true'
%run 07_fastapi_integration.ipynb
from httpx import ASGITransport, AsyncClient

async def verify_real_retail_api():
    base = f'/api/v1/projects/{REAL_RETAIL_PROJECT_ID}/market-basket'
    async with AsyncClient(transport=ASGITransport(app=app), base_url='http://testserver') as client:
        products_response = await client.get(base + '/products')
        assert products_response.status_code == 200, products_response.text
        loaded = products_response.json()
        assert loaded['transaction_count'] == 1000 and len(loaded['products']) == 40
        assert loaded['source_types'] == ['USER_DATASET'] and not loaded['is_synthetic']
        recommend_response = await client.post(base + '/recommend',
                                               json={'cart': [first_sku], **REAL_SETTINGS})
        assert recommend_response.status_code == 200, recommend_response.text
        recommendation = recommend_response.json()
        assert not recommendation['abstained']
        assert recommendation['training_transactions'] == 1000
        missing_response = await client.post(base + '/recommend',
                                              json={'cart': ['UNKNOWN-SKU'], **REAL_SETTINGS})
        assert missing_response.status_code == 200 and missing_response.json()['abstained']
        return recommendation

api_result = await verify_real_retail_api()
print('PASS: Existing Group 2 FastAPI returns real course data and valid recommendations')
print('API project ID:', REAL_RETAIL_PROJECT_ID)
print('Recommended SKU:', api_result['recommendations'][0]['sku'])
print('Training transactions:', api_result['training_transactions'])
print('All notebooks 00-19 remain unchanged.')

Database initialized: /Users/nehlinshanila/Downloads/group2-course-rag-engine/.runtime/group2.sqlite3
Document ingestion functions ready
Offline hybrid retrieval ready
Optional native pgvector adapter defined; credentials/database not required for local demo
Evidence-safe recommender ready
Research adapters and financial metrics ready
Batch research task runner ready
FastAPI routes: ['/openapi.json', '/docs', '/docs/oauth2-redirect', '/redoc', '/health', '/api/v1/projects/{project_id}/documents', '/api/v1/projects/{project_id}/documents', '/api/v1/projects/{project_id}/knowledge/search', '/api/v1/projects/{project_id}/recommendations/rank']
Database initialized: /Users/nehlinshanila/Downloads/group2-course-rag-engine/.runtime/group2.sqlite3
Apriori basket recommender ready; call seed_market_basket_demo() to load synthetic transactions.
Market basket REST routes registered in Group 2 FastAPI app
PASS: Existing Group 2 FastAPI returns real course data and valid recommendations
API projec